# Integrate Dynamic Knowledge Sources to Enhance RAG Output Quality
**IBM SkillsBuild Lab**

This notebook implements a RAG (Retrieval-Augmented Generation) system that integrates dynamic web-crawled content as its knowledge base.

**Stack:** BeautifulSoup (web crawl) → SentenceTransformers (embeddings) → FAISS (vector store) → IBM Granite via Replicate (LLM)

## Step 1 — Install Required Libraries

In [ ]:
!pip install sentence-transformers faiss-cpu langchain_community replicate

## Step 2 — Import Libraries

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_community.llms import Replicate
import os
from google.colab import userdata

## Step 3 — Set Replicate API Token

Store your token in Colab Secrets as `REPLICATE_api_token` (key icon in left sidebar), then run this cell.

In [ ]:
os.environ["REPLICATE_API_TOKEN"] = userdata.get('REPLICATE_api_token')
print("Replicate API token loaded.")

## Step 4 — Web Crawl: Extract Content from Dynamic Knowledge Source

Crawls `thealliance.ai`, strips scripts/styles, saves clean text to a file.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://thealliance.ai/"
try:
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # Remove script/style/noscript tags, then get visible text
    for s in soup(["script", "style", "noscript"]):
        s.decompose()

    global page_text
    page_text = soup.get_text(separator="\n")
    page_text = re.sub(r'\n\s*\n+', '\n\n', page_text)  # collapse multiple blank lines

    with open("thealliance_ai_content.txt", "w", encoding="utf-8") as f:
        f.write(page_text)

    print(f"Crawled {url} -> saved to 'thealliance_ai_content.txt' (chars: {len(page_text)})")

except Exception as e:
    print("Web crawl failed. You can create `page_text` manually. Error:", e)

## Step 5 — Chunk the Crawled Content

In [ ]:
def split_into_chunks(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

if 'page_text' in globals():
    chunks_web = split_into_chunks(page_text)
    print(f"Web-crawled content split into {len(chunks_web)} chunks.")
else:
    raise RuntimeError("No web-crawled text found. Run the crawl cell first.")

## Step 6 — Embed Chunks and Build FAISS Index

Uses `all-MiniLM-L6-v2` from SentenceTransformers to create dense embeddings, then indexes them in a FAISS `IndexFlatL2` for similarity search.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model (reuse if already present)
try:
    embed_model
except NameError:
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Loaded embedding model 'all-MiniLM-L6-v2'")

def build_index_from_chunks(chunks_local):
    embs = embed_model.encode(chunks_local, convert_to_numpy=True).astype(np.float32)
    dim = embs.shape[1]
    idx = faiss.IndexFlatL2(dim)
    idx.add(embs)
    return idx, embs

# Build web index if not already built
if 'chunks_web' in globals():
    if 'index_web' not in globals():
        index_web, embeddings_web = build_index_from_chunks(chunks_web)
        chunk_store_web = chunks_web
        print(f"Built FAISS index for web source ({len(chunks_web)} chunks).")
    else:
        print("Web FAISS index already present.")

## Step 7 — Enter Your Query

In [ ]:
try:
    user_query
except NameError:
    user_query = input("Enter your query: ")

print("The query you entered:", user_query)

## Step 8 — Retrieve Top-K Relevant Chunks

In [ ]:
import numpy as np

def retrieve_top_k(index_obj, chunk_store_local, query, k=3):
    q_emb = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, ids = index_obj.search(q_emb, k)
    top_chunks = [chunk_store_local[i] for i in ids[0]]
    return top_chunks, distances[0], ids[0]

retrieval_results = {}

if 'index_web' in globals() and 'chunk_store_web' in globals():
    top_chunks_web, dist_web, ids_web = retrieve_top_k(index_web, chunk_store_web, user_query, k=3)
    retrieval_results['web'] = {
        'top_chunks': top_chunks_web,
        'distances': dist_web,
        'ids': ids_web
    }
    print(f"Web: retrieved {len(top_chunks_web)} chunks.")

if not retrieval_results:
    raise RuntimeError("No index found. Run Step 6 to build indexes from chunks.")

## Step 9 — Review Retrieved Chunks

In [ ]:
for source in ['web']:
    if source in retrieval_results:
        print("\n" + "="*8 + f" Retrieved chunks — {source.upper()} " + "="*8 + "\n")
        for i, ch in enumerate(retrieval_results[source]['top_chunks'], start=1):
            print(f"[{i}] {ch[:500].replace(chr(10), ' ')}\n")